# `06_bicycle_route_infrastructure_per_side_per_spatial_unit`: Per-side bicycle route infrastructure per spatial unit

## Introduction

### Purpose

This notebook spatially joins `bicycle_route_infrastructure_per_side` (the per-side classified ways from `05_bicycle_route_infrastructure_per_side`) to municipality, province, and H3 boundaries. For each side–unit pair, the length of the facility portion falling within the unit is computed in EPSG:28992 and stored as `clipped_length_meters`. Per the thesis (§3.4.6), this is the stage-06 spatial-join step for the per-side branch; together with the parallel joins for the classified-route branch and the non-route layer, it produces the per-spatial-unit tables consumed by stage 07. The thesis reports 202,599 side-rows at the municipality level (§3.4.2), which is the expected output cardinality of the municipal join below.

> **`clipped_length_meters` is facility length, not road length.** For a carriageway way with infrastructure on both sides, both the left and right side-rows are clipped and each receives its own `clipped_length_meters`. Summing `clipped_length_meters` across all rows for a given carriageway way therefore yields bilateral facility length, which can be up to twice the road length. This is consistent with the facility-length accounting established in notebook 05 (thesis §3.4.2).

### Inputs

- `bicycle_route_infrastructure_per_side` from `05_bicycle_route_infrastructure_per_side`, loaded from the cached parquet (198,540 side-rows totalling 41,814.7 km of facility).
- `municipalities`, `provinces`, `h3_cells` from `03_boundaries_population`.
- `clip_ways_to_spatial_unit` helper from `functions.ipynb`.

### Outputs

- `bicycle_route_infrastructure_per_side_per_municipality`
- `bicycle_route_infrastructure_per_side_per_province`
- `bicycle_route_infrastructure_per_side_per_h3_cell`

Each row is one side–unit pair, carrying all per-side classification columns from notebook 05 (`side`, `cycleway_side`, `protection_level_per_side`, `unece_class`, `evidence_basis`, `classifiability`) plus `clipped_length_meters`.

### Key steps

The operation is structurally identical to the parallel stage-06 spatial joins for the classified-route branch (`06_bicycle_route_m_ways_distinct_classified_per_spatial_unit`) and the non-route layer (`06_non_bicycle_route_ways_per_spatial_unit`). Each side-row is intersected with spatial unit boundaries, split where it crosses multiple units, and assigned a `clipped_length_meters` computed from the clipped geometry in EPSG:28992. The only difference is the input grain: one row per way per side rather than one row per way. The notebook reads the input from the cached parquet produced by notebook 05 (rather than re-running notebook 05's unnesting via `%run`), which is a performance optimisation. Before clipping, the `way_m_geom` column is re-cast to the spatial extension's geometry type, since the parquet round-trip loses that typing information.

### Dependencies on prior notebooks

- `05_bicycle_route_infrastructure_per_side.ipynb` (output read from cache rather than re-run): provides `bicycle_route_infrastructure_per_side`.
- `03_boundaries_population.ipynb` (via `%run`): provides `municipalities`, `provinces`, `h3_cells`.
- `functions.ipynb` (via `%run`): provides the `clip_ways_to_spatial_unit` helper.

### Downstream consumers

- `07_bicycle_route_infrastructure_per_side_metrics` (stage 07 metrics): aggregates facility length by `protection_level_per_side`, `unece_class`, `evidence_basis`, and `classifiability` per spatial unit. This is the table the thesis headline results draw from: classifiable share, urban–rural gradient, evidence-basis decomposition, LISA clustering (thesis Results).

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Core clipping function](#2-core-clipping-function)
3. [Municipalities](#3-municipalities)
4. [Provinces](#4-provinces)
5. [H3 grid cells](#5-h3-grid-cells)
6. [Validation](#6-validation)

---

## 1. Environment setup

### Libraries and extensions

In [3]:
from IPython.utils import io
import duckdb
import geopandas

### Loading shared variables

`bicycle_route_infrastructure_per_side` is loaded directly from the cached parquet produced by notebook 05, which avoids re-running the expensive unnesting and per-side classification step. The boundary tables and the `clip_ways_to_spatial_unit` helper are brought in via `%run` of the relevant supporting notebooks.

In [1]:
from pathlib import Path
import duckdb

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

bicycle_route_infrastructure_per_side = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side.parquet"
)

In [4]:
with io.capture_output() as captured:
    %run /home/vbo226/03_boundaries_population.ipynb
    %run /home/vbo226/functions.ipynb  # for function 

In [7]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipalities_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

## 2. Core clipping function


## 2. Core clipping function

`clip_ways_to_spatial_unit` is defined in `06_non_bicycle_route_ways_per_spatial_unit` (the canonical home for the helper across all three stage-06 spatial-join notebooks) and is reused here without modification. In this notebook it is imported via `%run /home/vbo226/functions.ipynb` rather than via `%run` of the defining notebook directly.

The helper clips ways from any source table to any spatial-unit Arrow table, computing intersection length in EPSG:28992. The geometry column for `bicycle_route_infrastructure_per_side` is `way_m_geom` (the same column carried through from `bicycle_route_m_ways_distinct`), passed explicitly via `geom_col`.

## 3. Municipalities


## 3. Municipalities

Before the spatial join, the `way_m_geom` column is re-cast to the spatial extension's geometry type. The parquet round-trip in §1 stripped the spatial type from the column (it comes back as a generic blob), and `clip_ways_to_spatial_unit`'s `ST_Intersects` / `ST_Intersection` calls require the typed `GEOMETRY` representation. The cast is applied once to the in-memory table and reused by all three section joins below.

In [5]:
bicycle_route_infrastructure_per_side = duckdb.sql("SELECT * REPLACE(way_m_geom::geometry as way_m_geom) FROM bicycle_route_infrastructure_per_side")

In [8]:
bicycle_route_infrastructure_per_side_per_municipality = clip_ways_to_spatial_unit(
    spatial_unit_table = 'municipalities_arrow',
    ways_table         = 'bicycle_route_infrastructure_per_side',
    geom_col           = 'way_m_geom'
)
 
# Row count after clipping
bicycle_route_infrastructure_per_side_per_municipality.count('*')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       202599 │
└──────────────┘

---

## 4. Provinces

Clip the per-side classified ways to the 12 Dutch provinces.

In [28]:
bicycle_route_infrastructure_per_side_per_province = clip_ways_to_spatial_unit(
    spatial_unit_table = 'provinces_arrow',
    ways_table         = 'bicycle_route_infrastructure_per_side',
    geom_col           = 'way_m_geom'
)
 
# Row count after clipping
bicycle_route_infrastructure_per_side_per_province.count('*')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       199032 │
└──────────────┘

---

## 5. H3 grid cells

Clip the per-side classified ways to the 66,530 resolution-8 H3 cells covering the Netherlands. As in the parallel stage-06 joins, the finer H3 grid produces a substantially larger row-count multiplier than the administrative units.

In [29]:
bicycle_route_infrastructure_per_side_per_h3_cell = clip_ways_to_spatial_unit(
    spatial_unit_table = 'h3_cells_arrow',
    ways_table         = 'bicycle_route_infrastructure_per_side',
    geom_col           = 'way_m_geom'
)
 
# Row count after clipping
bicycle_route_infrastructure_per_side_per_h3_cell.count('*')

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       259708 │
└──────────────┘

---

## 6. Validation

Two checks are performed: row count and length conservation.

**Row count.** The spatial join increases the row count relative to `bicycle_route_infrastructure_per_side` wherever side-rows cross spatial-unit boundaries. Finer spatial units (H3) produce more fragmentation than coarser ones (provinces). The multiplier is expected to be higher than for the equivalent check on the classified-route branch because the input already has more rows: each carriageway way contributes two side-rows before the spatial join. The thesis reports 202,599 side-rows at the municipality level (§3.4.2), which is the expected value of the "After municipality join" row below.

**Length conservation.** The sum of `clipped_length_meters` across all rows for a given spatial-unit level should equal the sum of `way_m_length_nl_meters` across all side-rows in `bicycle_route_infrastructure_per_side`. Spatial intersection redistributes length across unit boundaries but does not create or destroy it. (A small shortfall is expected for the H3 join along the coastline, mirroring the shortfall observed in the two parallel stage-06 notebooks.)

After the row-count and conservation checks, the protection-level distribution at the municipality level is also reported. Every `protection_level_per_side` value present in `bicycle_route_infrastructure_per_side` should also appear here. A missing value would indicate that side-rows with that protection level were dropped by the spatial join (for instance, if they carried `NULL` geometries or fell entirely outside the municipal boundaries).

In [9]:
# Input: total facility km in bicycle_route_infrastructure_per_side
total_facility_km_input = duckdb.sql("""
    SELECT ROUND(SUM(way_m_length_nl_meters) / 1000, 3)
    FROM bicycle_route_infrastructure_per_side
""").fetchone()[0]
 
# After municipal join
municipality_count      = bicycle_route_infrastructure_per_side_per_municipality.count('*').fetchone()[0]
municipality_facility_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_infrastructure_per_side_per_municipality
""").fetchone()[0]
 
# After province join
province_count      = bicycle_route_infrastructure_per_side_per_province.count('*').fetchone()[0]
province_facility_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_infrastructure_per_side_per_province
""").fetchone()[0]
 
# After H3 join
h3_count      = bicycle_route_infrastructure_per_side_per_h3_cell.count('*').fetchone()[0]
h3_facility_km = duckdb.sql("""
    SELECT ROUND(SUM(clipped_length_meters) / 1000, 3)
    FROM bicycle_route_infrastructure_per_side_per_h3_cell
""").fetchone()[0]
 
# Input side-row count
n_sides = duckdb.sql("""
    SELECT COUNT(*) FROM bicycle_route_infrastructure_per_side
""").fetchone()[0]
 
print(f"{'':45s} {'rows':>10s}  {'multiplier':>10s}  {'facility km':>12s}")
print(f"{'─' * 82}")
print(f"{'Input (bicycle_route_infrastructure_per_side)':45s} {n_sides:>10,}  {'':>10s}  {total_facility_km_input:>12,.3f}")
print(f"{'After municipality join':45s} {municipality_count:>10,}  {municipality_count/n_sides:>10.3f}x  {municipality_facility_km:>12,.3f}")
print(f"{'After province join':45s} {province_count:>10,}  {province_count/n_sides:>10.3f}x  {province_facility_km:>12,.3f}")
print(f"{'After H3 join':45s} {h3_count:>10,}  {h3_count/n_sides:>10.3f}x  {h3_facility_km:>12,.3f}")
 
# Length conservation check
print(f"\nLength conservation (facility km should match input across all levels):")
print(f"  Municipality: {municipality_facility_km == total_facility_km_input}")
print(f"  Province:     {province_facility_km == total_facility_km_input}")
print(f"  H3:           {h3_facility_km == total_facility_km_input}")
 
# Protection level distribution is preserved after spatial join
# Check that all protection_level_per_side values present in input appear in output
duckdb.sql("""
SELECT
    protection_level_per_side,
    COUNT(*)                                                AS n_rows,
    ROUND(SUM(clipped_length_meters) / 1000, 3)            AS facility_km,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)     AS pct_rows
FROM bicycle_route_infrastructure_per_side_per_municipality
GROUP BY 1
ORDER BY n_rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

NameError: name 'bicycle_route_infrastructure_per_side_per_province' is not defined

### Export

Persist the three per-spatial-unit tables to disk so the stage-07 metrics notebook can read them without re-running the spatial join.

In [31]:
from pathlib import Path
import os

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)


def safe_write_parquet(df, final_path):
    tmp_path = final_path + ".tmp"

    df.write_parquet(tmp_path)
    os.replace(tmp_path, final_path)

safe_write_parquet(
    bicycle_route_infrastructure_per_side_per_municipality,
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_municipality.parquet"
)

safe_write_parquet(
    bicycle_route_infrastructure_per_side_per_province,
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_province.parquet"
)

safe_write_parquet(
    bicycle_route_infrastructure_per_side_per_h3_cell,
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_h3_cell.parquet"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))